# Generative Classification and Image Generation with Kernel Density Matrices

This notebook demonstrates **generative classification** using KDMs — a training mode
where the KDM prototypes jointly maximize classification accuracy **and** input data
likelihood. Because the prototypes are representative of the data distribution, the
same model can also be used as a **conditional generative model** to synthesize new
samples conditioned on a class label.

We show two examples:
1. **Two Moons** — a shallow generative classifier on 2-D synthetic data.
2. **MNIST** — a deep CNN encoder with a KDM inference layer, followed by decoder
   training and digit generation.

PyTorch port of the Keras 3 legacy notebook at `examples/legacy/kdm_classification_generation.ipynb`.

In [ ]:
# Uncomment to install the required packages
# !pip install kdm-torch torchvision

In [ ]:
import os
import math
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

import kdm
from kdm.models import KDMClassModel
from kdm.layers import KDMLayer, CosineKernelLayer
from kdm.utils import pure2dm, dm2comp
from kdm.init import init_kdm_layer

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
torch.manual_seed(0)
np.random.seed(0)

In [ ]:
# --------------- shared helpers ---------------

def plot_data(X, y):
    colors = plt.cm.rainbow(np.linspace(0, 1, len(np.unique(y))))
    for cls, color in zip(np.unique(y), colors):
        mask = y == cls
        plt.scatter(X[mask, 0], X[mask, 1], c=color, alpha=0.5, edgecolor='k',
                    label=f"Class {cls}")
    plt.legend(loc="best")


def plot_decision_region(X, pred_fun):
    pad = 0.05
    x0, x1 = X[:, 0].min() - pad, X[:, 0].max() + pad
    y0, y1 = X[:, 1].min() - pad, X[:, 1].max() + pad
    XX, YY = np.meshgrid(np.linspace(x0, x1, 50), np.linspace(y0, y1, 50))
    ZZ = pred_fun(np.c_[XX.ravel(), YY.ravel()]).reshape(XX.shape)
    plt.contourf(XX, YY, ZZ, 100, cmap=plt.cm.coolwarm, vmin=0, vmax=1)
    plt.colorbar()
    plt.contour(XX, YY, ZZ, levels=[0.1 * i for i in range(1, 10)])


@torch.no_grad()
def gen_pred_fun(model):
    def pred_fun(X):
        model.eval()
        probs = model(torch.as_tensor(X, dtype=torch.float32))
        return probs[:, 1].cpu().numpy()
    return pred_fun


@torch.no_grad()
def evaluate(model, X, y):
    model.eval()
    X_t = X if isinstance(X, torch.Tensor) else torch.as_tensor(X, dtype=torch.float32)
    y_t = y if isinstance(y, torch.Tensor) else torch.as_tensor(y, dtype=torch.long)
    probs = model(X_t)
    loss = F.nll_loss(torch.log(probs.clamp_min(1e-7)), y_t).item()
    acc = (probs.argmax(1) == y_t).float().mean().item()
    return loss, acc


def display_imgs(imgs, nrow=10, title=None):
    """Display up to `nrow` images. imgs: tensor (N,1,H,W) or (N,H,W)."""
    if isinstance(imgs, torch.Tensor):
        imgs = imgs.detach().cpu().numpy()
    if imgs.ndim == 4:
        imgs = imgs[:, 0]
    n = min(len(imgs), nrow)
    fig, axes = plt.subplots(1, n, figsize=(n * 1.2, 1.5))
    if n == 1:
        axes = [axes]
    if title:
        fig.suptitle(title)
    for ax, img in zip(axes, imgs[:n]):
        ax.imshow(img, cmap='gray', vmin=0, vmax=1)
        ax.axis('off')
    plt.tight_layout(pad=0.1)
    plt.show()


def latent_scatter(model, X, y, n=5000, title="Latent space"):
    """Plot 2-D latent encoding coloured by class label."""
    model.eval()
    X_t = X if isinstance(X, torch.Tensor) else torch.as_tensor(X, dtype=torch.float32)
    y_np = y.numpy() if isinstance(y, torch.Tensor) else y
    with torch.no_grad():
        enc = model.encoder(X_t[:n]).cpu().numpy()
    sc = plt.scatter(enc[:, 0], enc[:, 1], alpha=0.4, c=y_np[:n],
                     cmap='tab10', vmin=0, vmax=9, s=5)
    plt.colorbar(sc)
    plt.title(title)
    return enc

## Part 1 — Two Moons: Generative Classification

A standard `KDMClassModel` is a **discriminative** classifier: its prototypes are positioned
to maximize classification accuracy, with no regard for how well they cover the input
distribution.

**Generative co-training** adds a second objective — maximising the marginal log-likelihood
$\log P(x)$ of the input under the KDM's marginal distribution over $x$:

$$\mathcal{L}(x, y) = -\log P(y \mid x) - \alpha \log P(x)$$

With $\alpha=1$, both terms have equal weight. The prototypes end up distributed to cover
the data, not just the decision boundary. This is the key difference from the
pure discriminative case.

In [ ]:
X, y = make_moons(n_samples=1000, noise=0.2, random_state=0)
X_train2m, X_test2m, y_train2m, y_test2m = train_test_split(
    X, y, test_size=0.2, random_state=0
)
plot_data(X_train2m, y_train2m)
plt.title("Two Moons — training data")
plt.show()

In [ ]:
encoded_size = 2
dim_y = 2
n_comp = 14
alpha = 1.0  # generative weight

kdm_2m = KDMClassModel(
    encoded_size=encoded_size,
    dim_y=dim_y,
    encoder=nn.Identity(),
    n_comp=n_comp,
    sigma=0.05,
)

idx = np.random.randint(X_train2m.shape[0], size=n_comp)
init_kdm_layer(
    kdm_2m.kdm,
    encoded_x=torch.as_tensor(X_train2m[idx], dtype=torch.float32),
    samples_y=torch.as_tensor(np.eye(2)[y_train2m[idx]], dtype=torch.float32),
    init_sigma=True,
)

ds = TensorDataset(
    torch.as_tensor(X_train2m, dtype=torch.float32),
    torch.as_tensor(y_train2m, dtype=torch.long),
)
loader = DataLoader(ds, batch_size=32, shuffle=True)
opt = torch.optim.Adam(kdm_2m.parameters(), lr=5e-3)

for ep in range(25):
    kdm_2m.train()
    ep_loss = 0.0
    for xb, yb in loader:
        probs = kdm_2m(xb)
        disc_loss = F.nll_loss(torch.log(probs.clamp_min(1e-7)), yb)
        # generative term: maximise log P(x) under the KDM marginal
        rho_x = pure2dm(kdm_2m.encoder(xb))  # Identity encoder → pure2dm(xb)
        gen_loss = -alpha * kdm_2m.kdm.log_marginal(rho_x).mean()
        loss = disc_loss + gen_loss
        opt.zero_grad(); loss.backward(); opt.step()
        ep_loss += loss.item() * xb.size(0)
    ep_loss /= len(ds)
    if (ep + 1) % 5 == 0:
        print(f"epoch {ep+1:3d}  loss={ep_loss:.4f}")

In [ ]:
plot_decision_region(X, gen_pred_fun(kdm_2m))
plot_data(X_train2m, y_train2m)
c_x = kdm_2m.kdm.c_x.detach().cpu().numpy()
plt.scatter(c_x[:, 0], c_x[:, 1], c='y', marker='X', s=120, edgecolor='k',
            label='prototypes')
plt.legend(loc='upper right')
plt.title("Two Moons — generative classifier (α=1)")
plt.show()

train_loss, train_acc = evaluate(kdm_2m,
    torch.as_tensor(X_train2m, dtype=torch.float32),
    torch.as_tensor(y_train2m, dtype=torch.long))
test_loss, test_acc = evaluate(kdm_2m,
    torch.as_tensor(X_test2m, dtype=torch.float32),
    torch.as_tensor(y_test2m, dtype=torch.long))
print(f"Train: loss={train_loss:.4f}  acc={train_acc:.4f}")
print(f"Test:  loss={test_loss:.4f}  acc={test_acc:.4f}")
print(f"Sigma: {float(kdm_2m.kernel.sigma):.4f}")

The yellow prototypes are spread across the data manifold rather than clustering near
the decision boundary — they are positioned to both classify correctly **and** cover
the input distribution.

## Part 2 — MNIST: Discriminative Classification

We encode MNIST images into a 2-D latent space via a small CNN, then apply a KDM
inference layer to classify digits. A 2-D latent space lets us visualise both the
data distribution and the prototype positions directly.

This part is purely discriminative (no generative term) — it serves as the baseline
for Part 3.

In [ ]:
from torchvision import datasets, transforms

mnist_train_full = datasets.MNIST(
    root='~/.cache/mnist', train=True, download=True,
    transform=transforms.ToTensor()
)
mnist_test_ds = datasets.MNIST(
    root='~/.cache/mnist', train=False, download=True,
    transform=transforms.ToTensor()
)

# Use a 20% training subset for a quick demo; increase for paper-quality numbers.
n_train_full = len(mnist_train_full)
perm = np.random.RandomState(42).permutation(n_train_full)
n_keep = n_train_full // 5
train_idx = perm[: int(n_keep * 0.2)]
val_idx   = perm[int(n_keep * 0.2) : n_keep]

def stack(ds, idx):
    xs = torch.stack([ds[i][0] for i in idx])
    ys = torch.tensor([ds[i][1] for i in idx], dtype=torch.long)
    return xs, ys

X_train_t, y_train_t = stack(mnist_train_full, train_idx)
X_val_t,   y_val_t   = stack(mnist_train_full, val_idx)
X_test_t   = torch.stack([mnist_test_ds[i][0] for i in range(len(mnist_test_ds))])
y_test_t   = torch.tensor([mnist_test_ds[i][1] for i in range(len(mnist_test_ds))],
                           dtype=torch.long)

print(f"train={X_train_t.shape}  val={X_val_t.shape}  test={X_test_t.shape}")

In [ ]:
def create_encoder(base_depth: int, encoded_size: int) -> nn.Module:
    """Small CNN: (bs,1,28,28) → (bs, encoded_size)."""
    return nn.Sequential(
        nn.Conv2d(1, base_depth, 5, stride=1, padding=2),   nn.LeakyReLU(),
        nn.Conv2d(base_depth, base_depth, 5, stride=2, padding=2), nn.LeakyReLU(),
        nn.Conv2d(base_depth, 2*base_depth, 5, stride=1, padding=2), nn.LeakyReLU(),
        nn.Conv2d(2*base_depth, 2*base_depth, 5, stride=2, padding=2), nn.LeakyReLU(),
        nn.Conv2d(2*base_depth, 4*encoded_size, 7, stride=1, padding=0), nn.LeakyReLU(),
        nn.Flatten(),
        nn.Linear(4*encoded_size, encoded_size),
    )

In [ ]:
ENCODED_SIZE = 2
DIM_Y = 10
BASE_DEPTH = 32

# --- encoder warm-up with cross-entropy ---
encoder_disc = create_encoder(BASE_DEPTH, ENCODED_SIZE)
head = nn.Sequential(encoder_disc, nn.Linear(ENCODED_SIZE, DIM_Y))
opt_wu = torch.optim.Adam(head.parameters(), lr=1e-3)
ds_wu = TensorDataset(X_train_t, y_train_t)
loader_wu = DataLoader(ds_wu, batch_size=64, shuffle=True)
head.train()
for xb, yb in loader_wu:
    logits = head(xb)
    loss_wu = F.cross_entropy(logits, yb)
    opt_wu.zero_grad(); loss_wu.backward(); opt_wu.step()
print("Encoder warm-up done.")

# --- discriminative KDM model ---
N_COMP_DISC = 64

kdm_disc = KDMClassModel(
    encoded_size=ENCODED_SIZE, dim_y=DIM_Y, encoder=encoder_disc,
    n_comp=N_COMP_DISC, sigma=1.0, w_train=True,
)

idx = np.random.randint(X_train_t.shape[0], size=N_COMP_DISC)
encoder_disc.eval()
with torch.no_grad():
    enc_x = encoder_disc(X_train_t[idx])
init_kdm_layer(
    kdm_disc.kdm,
    encoded_x=enc_x,
    samples_y=F.one_hot(y_train_t[idx], DIM_Y).float(),
    init_sigma=True,
)

opt_disc = torch.optim.Adam(kdm_disc.parameters(), lr=1e-3)
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=128, shuffle=True)
history_disc = {'loss': [], 'val_loss': [], 'acc': [], 'val_acc': []}
EPOCHS_DISC = 5

for ep in range(EPOCHS_DISC):
    kdm_disc.train()
    for xb, yb in train_loader:
        probs = kdm_disc(xb)
        loss = F.nll_loss(torch.log(probs.clamp_min(1e-7)), yb)
        opt_disc.zero_grad(); loss.backward(); opt_disc.step()
    tr_loss, tr_acc = evaluate(kdm_disc, X_train_t, y_train_t)
    va_loss, va_acc = evaluate(kdm_disc, X_val_t[:2000], y_val_t[:2000])
    history_disc['loss'].append(tr_loss); history_disc['val_loss'].append(va_loss)
    history_disc['acc'].append(tr_acc);  history_disc['val_acc'].append(va_acc)
    print(f"epoch {ep+1:2d}  train={tr_loss:.4f}/{tr_acc:.3f}  val={va_loss:.4f}/{va_acc:.3f}")

test_loss, test_acc = evaluate(kdm_disc, X_test_t, y_test_t)
print(f"\nTest: loss={test_loss:.4f}  acc={test_acc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(history_disc['loss'], label='train')
axes[0].plot(history_disc['val_loss'], label='val')
axes[0].set_title('Loss'); axes[0].legend()
axes[1].plot(history_disc['acc'], label='train')
axes[1].plot(history_disc['val_acc'], label='val')
axes[1].set_title('Accuracy'); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
enc_disc = latent_scatter(kdm_disc, X_train_t, y_train_t, n=5000,
                          title="Discriminative — latent space")
protos = kdm_disc.kdm.c_x.detach().cpu().numpy()
plt.scatter(protos[:, 0], protos[:, 1], c='k', marker='X', s=80,
            label='prototypes')
plt.legend()
plt.show()

The black prototypes concentrate near decision boundaries, leaving much of the
data manifold uncovered. This makes direct sampling from the model unreliable.

## Part 3 — MNIST: Generative Classification

We add the generative term $-\alpha \log P(x)$ to the loss, computed via
`KDMLayer.log_marginal`. We reuse the encoder weights from Part 2 and **freeze
the encoder** — only the KDM prototypes and kernel sigma are updated. This forces
the prototypes to reposition themselves to cover the data distribution within the
fixed latent space learned by the discriminative encoder.

In [ ]:
N_COMP_GEN = 128
ALPHA = 1.0         # generative weight
EPOCHS_GEN = 10

# New model: transfer encoder weights + sigma from discriminative model.
encoder_gen = create_encoder(BASE_DEPTH, ENCODED_SIZE)
encoder_gen.load_state_dict(encoder_disc.state_dict())

kdm_gen_cls = KDMClassModel(
    encoded_size=ENCODED_SIZE, dim_y=DIM_Y, encoder=encoder_gen,
    n_comp=N_COMP_GEN, sigma=kdm_disc.kernel.sigma.item(), w_train=True,
)

idx = np.random.randint(X_train_t.shape[0], size=N_COMP_GEN)
encoder_gen.eval()
with torch.no_grad():
    enc_x_gen = encoder_gen(X_train_t[idx])
init_kdm_layer(
    kdm_gen_cls.kdm,
    encoded_x=enc_x_gen,
    samples_y=F.one_hot(y_train_t[idx], DIM_Y).float(),
    init_sigma=False,  # reuse sigma from discriminative model
)

# Freeze encoder: only the KDM layer and kernel sigma are trained.
for p in encoder_gen.parameters():
    p.requires_grad_(False)

opt_gen = torch.optim.Adam(
    filter(lambda p: p.requires_grad, kdm_gen_cls.parameters()), lr=1e-3
)
history_gen = {'loss': [], 'val_loss': [], 'acc': [], 'val_acc': []}

for ep in range(EPOCHS_GEN):
    kdm_gen_cls.train()
    for xb, yb in train_loader:
        probs = kdm_gen_cls(xb)
        disc_loss = F.nll_loss(torch.log(probs.clamp_min(1e-7)), yb)
        # generative term: log P(x) under the learned KDM marginal over x
        rho_x = pure2dm(kdm_gen_cls.encoder(xb))
        gen_loss = -ALPHA * kdm_gen_cls.kdm.log_marginal(rho_x).mean()
        loss = disc_loss + gen_loss
        opt_gen.zero_grad(); loss.backward(); opt_gen.step()
    tr_loss, tr_acc = evaluate(kdm_gen_cls, X_train_t, y_train_t)
    va_loss, va_acc = evaluate(kdm_gen_cls, X_val_t[:2000], y_val_t[:2000])
    history_gen['loss'].append(tr_loss); history_gen['val_loss'].append(va_loss)
    history_gen['acc'].append(tr_acc);  history_gen['val_acc'].append(va_acc)
    print(f"epoch {ep+1:2d}  train={tr_loss:.4f}/{tr_acc:.3f}  val={va_loss:.4f}/{va_acc:.3f}")

test_loss, test_acc = evaluate(kdm_gen_cls, X_test_t, y_test_t)
print(f"\nTest: loss={test_loss:.4f}  acc={test_acc:.4f}")
print(f"Sigma: {kdm_gen_cls.kernel.sigma.item():.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plt.sca(axes[0])
enc_disc_plot = latent_scatter(kdm_disc, X_train_t, y_train_t, n=5000,
                               title="Discriminative prototypes")
protos_disc = kdm_disc.kdm.c_x.detach().cpu().numpy()
plt.scatter(protos_disc[:, 0], protos_disc[:, 1], c='k', marker='X', s=60)

plt.sca(axes[1])
latent_scatter(kdm_gen_cls, X_train_t, y_train_t, n=5000,
               title="Generative prototypes")
protos_gen = kdm_gen_cls.kdm.c_x.detach().cpu().numpy()
plt.scatter(protos_gen[:, 0], protos_gen[:, 1], c='k', marker='X', s=60)

plt.tight_layout(); plt.show()

The generative prototypes (right) spread more uniformly across each class region,
making them suitable for sampling representative points from the distribution.

## Part 4 — Autoencoder: Training a Decoder

To generate pixel-space images we need a **decoder** that maps a 2-D latent code back
to a 28×28 image. We train a convolutional decoder jointly with the (frozen) encoder
as a standard autoencoder.

In [ ]:
def create_decoder(encoded_size: int, base_depth: int) -> nn.Module:
    """Transpose-conv decoder: (bs, encoded_size) → (bs, 1, 28, 28) logits.

    Shape trace (encoded_size=2, base_depth=32):
      (bs,2) → Unflatten → (bs,2,1,1)
      ConvT(7, s=1, p=0) → (bs,64,7,7)
      ConvT(4, s=2, p=1) → (bs,64,14,14)
      ConvT(4, s=2, p=1) → (bs,32,28,28)
      Conv (3, s=1, p=1) → (bs,1,28,28)
    """
    return nn.Sequential(
        nn.Unflatten(1, (encoded_size, 1, 1)),
        nn.ConvTranspose2d(encoded_size, 2*base_depth, 7, stride=1, padding=0),
        nn.LeakyReLU(),
        nn.ConvTranspose2d(2*base_depth, 2*base_depth, 4, stride=2, padding=1),
        nn.LeakyReLU(),
        nn.ConvTranspose2d(2*base_depth, base_depth, 4, stride=2, padding=1),
        nn.LeakyReLU(),
        nn.Conv2d(base_depth, 1, 3, stride=1, padding=1),
    )

In [ ]:
decoder = create_decoder(ENCODED_SIZE, BASE_DEPTH)

# Freeze the generative encoder during decoder training.
for p in encoder_gen.parameters():
    p.requires_grad_(False)
encoder_gen.eval()

opt_ae = torch.optim.Adam(decoder.parameters(), lr=1e-4)
ae_loader = DataLoader(TensorDataset(X_train_t), batch_size=128, shuffle=True)
EPOCHS_AE = 20

for ep in range(EPOCHS_AE):
    decoder.train()
    ep_loss = 0.0
    for (xb,) in ae_loader:
        with torch.no_grad():
            z = encoder_gen(xb)
        recon_logits = decoder(z)
        loss = F.binary_cross_entropy_with_logits(recon_logits, xb)
        opt_ae.zero_grad(); loss.backward(); opt_ae.step()
        ep_loss += loss.item() * xb.size(0)
    ep_loss /= len(X_train_t)
    if (ep + 1) % 5 == 0:
        print(f"epoch {ep+1:3d}  recon_loss={ep_loss:.4f}")

# Unfreeze encoder for potential further use.
for p in encoder_gen.parameters():
    p.requires_grad_(True)

In [ ]:
# Visualise original vs reconstructed images.
n_show = 10
encoder_gen.eval(); decoder.eval()
with torch.no_grad():
    z = encoder_gen(X_test_t[:n_show])
    recon = torch.sigmoid(decoder(z))

display_imgs(X_test_t[:n_show], nrow=n_show, title="Originals")
display_imgs(recon, nrow=n_show, title="Reconstructions")

Reconstructions are blurry because the 2-D latent space must compress all the
structural variation of MNIST into just two coordinates. A higher-dimensional latent
space would yield sharper images.

## Part 5 — Conditional Image Generation

The KDM inference layer computes a **conditional** mapping $x \mapsto y$. By symmetry,
swapping the input and output prototypes ($c_x \leftrightarrow c_y$) reverses the mapping
to $y \mapsto x$. Combined with the cosine kernel (suited for normalised probability
vectors), this gives a **generator** that maps a class distribution to a distribution
over latent codes.

We then decode latent samples through the trained decoder to obtain pixel-space images.

In [ ]:
def create_generator(clf_model: KDMClassModel) -> KDMLayer:
    """Build a KDM generator by swapping the input/output prototypes.

    The classifier maps x (encoded_size-D) → y (dim_y-D).
    The generator maps y (dim_y-D) → x (encoded_size-D).
    """
    n_comp = clf_model.kdm.c_x.shape[0]
    dim_x_gen = clf_model.kdm.c_y.shape[1]  # = dim_y of classifier = 10
    dim_y_gen = clf_model.kdm.c_x.shape[1]  # = encoded_size = 2
    kdm_gen = KDMLayer(
        kernel=CosineKernelLayer(),
        dim_x=dim_x_gen,
        dim_y=dim_y_gen,
        n_comp=n_comp,
    )
    with torch.no_grad():
        kdm_gen.c_x.copy_(clf_model.kdm.c_y)  # class prototypes → input
        kdm_gen.c_y.copy_(clf_model.kdm.c_x)  # latent prototypes → output
        kdm_gen.c_w.copy_(clf_model.kdm.c_w)
    return kdm_gen


kdm_generator = create_generator(kdm_gen_cls)

In [ ]:
def dm2distrib(dm, sigma):
    """Convert a KDM (batch of GMMs) to a torch MixtureSameFamily distribution.

    The RBF kernel in KDM corresponds to Gaussian components with std sigma/sqrt(2).
    """
    w, v = dm2comp(dm)  # w: (bs, K), v: (bs, K, d)
    return torch.distributions.MixtureSameFamily(
        torch.distributions.Categorical(probs=w),
        torch.distributions.Independent(
            torch.distributions.Normal(v, sigma / math.sqrt(2)), 1
        ),
    )


@torch.no_grad()
def generate_class(digit: int, n_samples: int = 10, sigma: float | None = None):
    """Sample `n_samples` images conditioned on `digit`."""
    if sigma is None:
        sigma = kdm_gen_cls.kernel.sigma.item()
    y_one_hot = F.one_hot(torch.tensor([digit]), DIM_Y).float()  # (1, 10)
    rho_in = pure2dm(y_one_hot)                                  # (1, 1, 11)
    rho_out = kdm_generator(rho_in)                              # (1, K, 3)
    distrib = dm2distrib(rho_out, sigma)
    latent = distrib.sample((n_samples,)).squeeze(1)             # (n_samples, 2)
    decoder.eval()
    imgs = torch.sigmoid(decoder(latent))                        # (n_samples, 1, 28, 28)
    return imgs

In [ ]:
# Generate 10 samples for each digit class (0–9).
for digit in range(10):
    imgs = generate_class(digit, n_samples=10)
    display_imgs(imgs, nrow=10, title=f"Generated digit: {digit}")

In [ ]:
# Visualise the latent samples for each class alongside the full embedding.
encoder_gen.eval()
with torch.no_grad():
    enc_gen = encoder_gen(X_train_t[:5000]).cpu().numpy()

plt.figure(figsize=(8, 6))
plt.scatter(enc_gen[:, 0], enc_gen[:, 1], alpha=0.3, c=y_train_t[:5000].numpy(),
            cmap='tab10', vmin=0, vmax=9, s=5, label='training data')

colors = plt.cm.tab10(np.linspace(0, 1, 10))
for digit in range(10):
    y_one_hot = F.one_hot(torch.tensor([digit]), DIM_Y).float()
    with torch.no_grad():
        rho_out = kdm_generator(pure2dm(y_one_hot))
    distrib = dm2distrib(rho_out, kdm_gen_cls.kernel.sigma.item())
    samples = distrib.sample((30,)).squeeze(1).numpy()
    plt.scatter(samples[:, 0], samples[:, 1], marker='x', s=60,
                color=colors[digit], linewidths=1.5)

plt.title("Latent space with generated samples (×) per class")
plt.colorbar(plt.cm.ScalarMappable(cmap='tab10'), label='class')
plt.show()

In [ ]:
# Interpolate between two classes: a superposition of class 0 and class 1.
@torch.no_grad()
def generate_mixture(digits, weights, n_samples=10, sigma=None):
    """Generate from a superposition of class labels."""
    if sigma is None:
        sigma = kdm_gen_cls.kernel.sigma.item()
    w = torch.tensor(weights, dtype=torch.float32)
    w = w / w.norm()  # L2-normalise for cosine kernel
    y_mix = torch.zeros(1, DIM_Y)
    for d, wt in zip(digits, w.tolist()):
        y_mix[0, d] = wt
    rho_in = pure2dm(y_mix)             # (1, 1, 11)
    rho_out = kdm_generator(rho_in)     # (1, K, 3)
    distrib = dm2distrib(rho_out, sigma)
    latent = distrib.sample((n_samples,)).squeeze(1)
    decoder.eval()
    return torch.sigmoid(decoder(latent))


imgs_mix = generate_mixture([0, 1], [1.0, 1.0], n_samples=10)
display_imgs(imgs_mix, nrow=10, title="Mixture: class 0 + class 1 (equal weight)")